# LAr optical map: mixup and last-layer fitting

Test whether fitting the final output layer on real simulation events reduces the bias of the CNP mean while retaining useful event ranking.

- **Development data:** LF1500, including zero-hit runs; 80% training and 20% validation voxels.
- **External test:** LF1000, including zero-hit runs. Locations within 5 mm of any development voxel are excluded.
- **Stage 1:** 12,000 class-aware mixup steps, with mixed context and independent mixing weights.
- **Stage 2:** 2,000 steps on real context and targets, using natural sampling and unweighted Bernoulli loss. Only the decoder’s final linear layer is trainable.

Validation selects checkpoints separately in each stage. The external test never selects a checkpoint. Both stages are evaluated on identical real context/target splits; observed fractions use the target events remaining after 64 context events.

The output remains an event score averaged within each voxel. Final-layer fitting does not guarantee calibrated probabilities. The two decoder channels retain the existing score calculation; their scale is not an independently validated uncertainty estimate.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import html

import torch
from IPython.display import HTML, Image, display
from schemas.optical_transfer import load_optical_transfer_config
from schemas.surrogates import load_surrogate_config
from data.optical_transfer import prepare_optical_transfer_data
from core.surrogates.experiment import run_experiment

torch.set_num_threads(1)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config.optical.fine_tuning.yaml").exists())
DATA_CONFIG = ROOT / "config.optical.fine_tuning.data.yaml"
TRAINING_CONFIG = ROOT / "config.optical.fine_tuning.yaml"
RUN_TRAINING = True

def show_table(rows, columns):
    cells = lambda row: "".join(f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns)
    header = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join(f"<tr>{cells(row)}</tr>" for row in rows)
    display(HTML(f"<table><tr>{header}</tr>{body}</table>"))


## Prepare the data

The existing optical reader, feature construction and voxel splitting are reused. Normalization is fitted only on the LF1500 training split. The data configuration contains the source folders, budgets and spatial holdout distance. Simulation random-seed independence must be established from simulator provenance; directory names alone cannot establish it.


In [ ]:
data_config = load_optical_transfer_config(DATA_CONFIG)
prepared = prepare_optical_transfer_data(data_config)
with (data_config.output_directory / "transfer_data_counts.csv").open() as file:
    counts = list(csv.DictReader(file))
show_table(counts, list(counts[0]))
audit = json.loads((data_config.output_directory / "test_source_manifest.json").read_text())
print(f"External test files excluded: {len(audit['excluded'])}")


## Train both stages

Steps count randomly sampled episodes, rather than full passes through the dataset. Stage 1 uses 128 events per episode; stage 2 uses 512 to expose more real positives per update. Learning rates are 0.001 and 0.0003 respectively, with 16 voxels per batch and evaluation every 250 steps. These are starting settings, selected checkpoints should be assessed on the validation history.

Set `RUN_TRAINING = False` to display the latest saved completed run. Each new training run gets its own output directory. The selected final checkpoint is `checkpoints/best`; the selected pre-fine-tuning checkpoint is `checkpoints/pretraining`.


In [ ]:
config = load_surrogate_config(TRAINING_CONFIG)
config.data_directory = data_config.output_directory
runs = config.output_directory.parent
if RUN_TRAINING:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    config.output_directory = runs / stamp
    output = run_experiment(config)
else:
    output = max(p.parent for p in runs.glob("*/metrics.json"))
print(output)


## Before and after

Compare the selected mixup checkpoint with the selected fine-tuned checkpoint on the same events. Mean bias is predicted minus observed. Training metrics are in-sample; validation selected checkpoints; the LF1000 test is held out.


In [ ]:
metrics = json.loads((output / "metrics.json").read_text())
comparison = [r for r in metrics if r["checkpoint"] in ("pretraining", "best")]
show_table(comparison, ["split", "checkpoint", "voxels", "positives", "mean_observed",
                        "mean_predicted", "mean_bias", "voxel_rate_mae", "pearson_r",
                        "average_precision", "bernoulli_log_loss"])
for filename in ["validation_history.png", "lf_validation_fine_tuning_means.png",
                 "lf_test_fine_tuning_means.png", "lf_train_fine_tuning_pr.png",
                 "lf_test_fine_tuning_pr.png"]:
    display(Image(filename=str(output / filename)))


## Integration with the spatial GP

This experiment isolates the CNP training change. The saved checkpoint implements the standard interchangeable-model API, so its averaged scores can enter the existing MFGP stage. In a full RESuM run, adding `training.fine_tuning` to the surrogate configuration makes the existing MFGP stage consume the selected fine-tuned model. Removing that block restores single-stage training.
